In [0]:
import pyspark.sql.functions as F


df = spark.read.table("retaildataplatform.bronze.sqlserver_customers")

"cast_columns": {

        "customer_id": "long",

        "tax_id": "string",

        "tax_code": "string",

        "city": "string"
        }

"rename_columns": {
        "customer_id": "CustomerId"
    }

PascalCase
snake_case
def transform_customer(df):

    df = df.withColumn("customer_name", F.lower(df["customer_name"]))\
        .withColumn("name_parts", F.split(F.col("customer_name"),","))\
        .withColumn("first_name", F.expr("get(name_parts,1)"))\
        .withColumn("last_name", F.expr("get(name_parts,0)"))\
        .withColumn("city", F.upper(df["city"]))\
        .withColumn("postcode", F.regexp_replace(F.col("postcode"), "\.0$", ""))\
        .withColumn("valid_from", F.from_unixtime(F.expr("try_cast(valid_from as bigint)"))) \
        .withColumn("valid_to", F.from_unixtime(F.expr("try_cast(valid_to as bigint)")))\
        .withColumn('country',F.lit('USA'))

    df = df.drop("name_parts","customer_name","file_path")

    df = df.dropDuplicates(["customer_id"])

    return df.select(
        "customer_id",
        "first_name",
        "last_name",
        "tax_id",
        "tax_code",
        "country",
        "state",
        "city",
        "postcode",
        "street",
        "number",
        "unit",
        "region",
        "district",
        "lon",
        "lat",
        "ship_to_address",
        "valid_from",
        "valid_to",
        "units_purchased",
        "loyalty_segment",
        "last_update_ts"
    )

df = spark.read.table("retaildataplatform.bronze.sqlserver_customers")
cleaned_customer = transform_customer(df)

target_table = "retaildataplatform.silver.customers"

if spark.catalog.tableExists("retaildataplatform.silver.customers"):
    print("Table Exists - Now Proceeding with SCD 1")

    # Load Target

    target = DeltaTable.forName(spark, target_table)

    # SCD Type 1

    # Match in source and target -> Key exist in target
    (
        target
        .merge(
            cleaned_customer.alias("source"), 
            "target.customer_id = source.customer_id"

        )


    # Update - Match
    .whenMatchedUpdateAll()

    # Insert - Not Match
    .whenNotMatchedInsertAll()

    .execute()
    )


else:
    print("Table Does Not Exist - Now Creating Table")
    spark.sql("""Create Schema if not exists silver """)
    cleaned_customer.write.format("delta").mode("overwrite").saveAsTable("retaildataplatform.silver.customers")




# Common Functions for 

- SCD 1 -> scdfunctions.py 


- rename_columns, safe_cast, drop -> generic.py



use it in all 3 silver layer notebooks


- customer and sales_orders -> Modularise ( with different config file + proper logging + use of necessary common_utility functions)

- sales notebook from scratch
